In [5]:
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.utils.extmath import randomized_svd

In [6]:
INPUT_CSV  = "../dane/fra_cleaned_final.csv"
OUTPUT_CSV = "../dane/fra_cleaned_final_embeddings.csv"

NOTE_MAP = {}

CONFIG = dict(
    LAYER_WEIGHTS = {"Top": 1.0, "Middle": 1.0, "Base": 1.0}, # Równa waga dla każdej warstwy piramidy
    LENGTH_NORM_EXP = 0.0,                                    # Brak kary za długość listy składników
    MIN_DF_NOTE = 3,                                          # Nuta musi wystąpić min. 3 razy w całej bazie
    ACCORD_RANK_WEIGHTS = [1.0, 0.75, 0.56, 0.42, 0.32],      # Malejące wagi dla kolejnych głównych akordów, 0.75^k
    IDF_POWER = 0.5,                                          # Pierwiastek wygładzający wagę rzadkich nut, osłabia nuty dominujące np. piżmo
    A_ALPHA_ACCORD = 0.4,                                     # Metoda A: 40% wpływu akordów, 60% nut
    B_CTX_SMOOTH = 0.5,                                       # Metoda B: Wygładzanie korelacji między nutami, 1 - brak wygładzania, 0 - pełne wygładzenie
    B_ACCORD_GAIN = 1.5,                                      # Metoda B: Bonus dla zapachów z tej samej rodziny
    DIM = 256,                                                # Docelowa liczba wymiarów wektora zapachu (kompresja SVD)
    SEED = 42,                                                # Stałe ziarno losowości dla powtarzalności wyników
)

In [16]:
df = pd.read_csv(INPUT_CSV, sep=";", encoding="utf-8-sig")
N = len(df)  

print("kształt tabeli (wiersze, kolumny):", df.shape)
print("\nnazwy kolumn:")
print(df.columns.tolist())

print("\npierwsze trzy rekordy:")
print(df.loc[23577].T)

braki = df.isna().sum()
print("\nbraki danych:")
print(braki[braki > 0])

print("\npowtórzone adresy url:", df.url.duplicated().sum())
print("powtórzone pary (Perfume, Brand):", df.duplicated(["Perfume", "Brand"]).sum())

kształt tabeli (wiersze, kolumny): (24046, 17)

nazwy kolumn:
['url', 'Perfume', 'Brand', 'Country', 'Gender', 'Rating Value', 'Rating Count', 'Year', 'Top', 'Middle', 'Base', 'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5', 'Perfumers']

pierwsze trzy rekordy:
url             https://www.fragrantica.com/perfume/issey-miya...
Perfume                       L EAU D ISSEY POUR HOMME NOIR AMBRE
Brand                                                ISSEY MIYAKE
Country                                                     Japan
Gender                                                        men
Rating Value                                                  4.5
Rating Count                                                 1044
Year                                                       2016.0
Top                          saffron, cinnamon, nutmeg, coriander
Middle                           leather, cedar, papyrus, vetiver
Base                                   amber, vanill

In [17]:
def split_list(s):
    if not isinstance(s, str):
        return []
    return [x.strip().lower() for x in s.split(",")
            if x.strip() and x.strip().lower() != "unknown"]

for layer in ["Top", "Middle", "Base"]:
    df[layer + "_l"] = df[layer].map(split_list)

acc_cols = [f"mainaccord{i}" for i in range(1, 6)]
df["accords_l"] = df[acc_cols].apply(
    lambda row: [str(x).strip().lower() for x in row if isinstance(x, str) and x.strip()],
    axis=1,
)

NOTE_MAP = {}   # do zrobienia mapowanie nut szczegolowych na bardziej ogolne np. {"bourbon vanilla": "vanilla"}

for layer in ["Top", "Middle", "Base"]:
    df[layer + "_l"] = df[layer + "_l"].map(lambda lst: [NOTE_MAP.get(n, n) for n in lst])

all_notes = Counter(n for layer in ["Top_l", "Middle_l", "Base_l"]
                    for lst in df[layer] for n in lst)
all_acc = Counter(a for lst in df["accords_l"] for a in lst)
counts = np.array(list(all_notes.values()))

print(f"unikalne nuty: {len(all_notes)}")
print(f"  występujące tylko raz: {(counts == 1).sum()}")
print(f"  występujące co najmniej 3 razy: {(counts >= 3).sum()}")
print(f"  występujące co najmniej 50 razy: {(counts >= 50).sum()}")
print(f"unikalne akordy: {len(all_acc)}")

print("\nśrednia liczba nut w warstwie:",
      {l: round(float(df[l + "_l"].map(len).mean()), 2) for l in ["Top", "Middle", "Base"]})
print("\nile perfum ma ile akordów:")
print(df["accords_l"].map(len).value_counts().sort_index().to_string())
print("\nnajczęstsze nuty:", all_notes.most_common(10))

print("\nprzykład:")
print(df.loc[23577, ["Perfume", "Top", "Top_l", "accords_l"]].to_string())

unikalne nuty: 1664
  występujące tylko raz: 343
  występujące co najmniej 3 razy: 1135
  występujące co najmniej 50 razy: 366
unikalne akordy: 84

średnia liczba nut w warstwie: {'Top': 3.11, 'Middle': 3.3, 'Base': 3.5}

ile perfum ma ile akordów:
accords_l
1       13
2      101
3      274
4      591
5    23067

najczęstsze nuty: [('musk', 11013), ('bergamot', 8613), ('sandalwood', 8057), ('jasmine', 7741), ('amber', 7725), ('patchouli', 7250), ('vanilla', 6738), ('rose', 6249), ('cedar', 5625), ('mandarin orange', 4006)]

przykład:
Perfume               L EAU D ISSEY POUR HOMME NOIR AMBRE
Top                  saffron, cinnamon, nutmeg, coriander
Top_l              [saffron, cinnamon, nutmeg, coriander]
accords_l    [amber, leather, warm spicy, vanilla, woody]


In [ ]:
# 1) cechy-nuty: każda nuta dostaje wagę 1 za każde wystąpienie (głowa, serce i baza ważą tyle samo, długość piramidy nie ma znaczenia)
def note_features():
    wynik = []
    for top, middle, base in zip(df.Top_l, df.Middle_l, df.Base_l):
        cechy = {}
        for lista in (top, middle, base):
            for nuta in lista:
                cechy[nuta] = cechy.get(nuta, 0.0) + 1.0
        wynik.append(cechy)
    return wynik

# 2) cechy-akordy: waga zależy od pozycji (1., 2., 3., 4., 5.)
def accord_features():
    wagi = CONFIG["ACCORD_RANK_WEIGHTS"]
    return [{akord: wagi[i] for i, akord in enumerate(lista[:5])}
            for lista in df.accords_l]

# 3) zamiana słowników cech na oszczędną tabelę (macierz rzadką)
def to_sparse(cechy_perfum, min_df=1):
    # ile perfum ma daną cechę
    licznik = Counter(k for d in cechy_perfum for k in d)
    # słownik: zostają tylko cechy występujące w co najmniej min_df perfumach
    vocab = sorted(k for k, v in licznik.items() if v >= min_df)
    indeks = {k: i for i, k in enumerate(vocab)}

    wiersze, kolumny, wartosci = [], [], []
    for i, d in enumerate(cechy_perfum):
        for k, v in d.items():
            j = indeks.get(k)
            if j is not None:
                wiersze.append(i); kolumny.append(j); wartosci.append(v)

    X = sp.csr_matrix((wartosci, (wiersze, kolumny)),
                      shape=(len(cechy_perfum), len(vocab)))
    df_cechy = np.array([licznik[k] for k in vocab])   # w ilu perfumach jest każda cecha
    return X, vocab, df_cechy

# 4) stworzenie obu tabel cech (nuty i akordy) w formie macierzy rzadkich
X_note, vocab_note, df_note = to_sparse(note_features(), CONFIG["MIN_DF_NOTE"])
X_acc,  vocab_acc,  df_acc  = to_sparse(accord_features())

print("tabela nut:    ", X_note.shape)
print("tabela akordów:", X_acc.shape)
print(f"średnio cech na perfumy: nuty {X_note.nnz / N:.1f}, akordy {X_acc.nnz / N:.1f}")
print(f"wypełnienie tabeli nut: {X_note.nnz / (X_note.shape[0] * X_note.shape[1]):.2%}")

# 5) podgląd
i = 23577
def pokaz(X, vocab, i):
    wiersz = X[i]
    return {vocab[j]: round(float(v), 2) for j, v in zip(wiersz.indices, wiersz.data)}

print("\nROUGE IMPERIALE, nuty:  ", pokaz(X_note, vocab_note, i))
print("ROUGE IMPERIALE, akordy:", pokaz(X_acc, vocab_acc, i))

# 6) kontrola: najpopularniejsze cechy
print("\nnajczęstsze nuty (w ilu perfumach):",
      [(vocab_note[j], int(df_note[j])) for j in np.argsort(-df_note)[:5]])
print("najczęstsze akordy (w ilu perfumach):",
      [(vocab_acc[j], int(df_acc[j])) for j in np.argsort(-df_acc)[:5]])